# Findings and limitations | Issue #21

This notebook synthesizes the September EDA evidence into the canonical findings register. It independently recalculates the required checks from the frozen source, separates observed evidence from October modeling ideas, and publishes the Gate 3 verification bundle.

The automated comparisons below are not a substitute for independent human review. Reviewer identity, execution date, and approval remain pending until a named non-author reruns and signs the work.

In [1]:
from pathlib import Path
import hashlib
import itertools
import json
import platform
import subprocess

import numpy as np
import pandas as pd


def find_repo_root():
    for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (path / 'data' / 'allstate_claims_data.csv').is_file():
            return path
    raise FileNotFoundError('Repository root not found.')


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def relative(path):
    return Path(path).resolve().relative_to(ROOT).as_posix()


def executable_notebook_sha256(path):
    notebook = json.loads(Path(path).read_text(encoding='utf-8'))
    code_sources = [
        ''.join(cell.get('source', []))
        for cell in notebook['cells']
        if cell.get('cell_type') == 'code'
    ]
    payload = json.dumps(code_sources, ensure_ascii=False, separators=(',', ':'))
    return hashlib.sha256(payload.encode('utf-8')).hexdigest()


ROOT = find_repo_root()
SOURCE_PATH = ROOT / 'data' / 'allstate_claims_data.csv'
GATE2 = ROOT / 'notebooks/final-deliverables/September/Gate 2'
GATE3 = ROOT / 'notebooks/final-deliverables/September/Gate 3'
EVIDENCE_DIR = GATE3 / 'findings-and-limitations'
REGISTER_PATH = GATE2 / 'findings_register.csv'
REVIEW_REGISTER_PATH = GATE2 / 'independent_review_register.csv'
NOTEBOOK_PATH = ROOT / 'notebooks/ragib-nehal/ragib_nehal_task_13.ipynb'
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_SOURCE_SHA256 = '74037cb248a1064e4d578692a4f4e5d8492ed1b2033daf643496e1b68b14ae03'
REGISTER_VERSION = '2.0.0'
CONTINUOUS = [f'cont{i}' for i in range(1, 15)]
CATEGORICAL = [f'cat{i}' for i in range(1, 117)]

print('Repository root located; published paths are repository-relative.')
print('Evidence directory:', relative(EVIDENCE_DIR))

Repository root located; published paths are repository-relative.
Evidence directory: notebooks/final-deliverables/September/Gate 3/findings-and-limitations


## 1. Frozen-source verification

Verify the supplied dataset before using it. Completeness and uniqueness reduce repair work, but they do not remove the need to study distributions, categorical support, or validation design.

In [2]:
source_sha256 = sha256(SOURCE_PATH)
assert source_sha256 == EXPECTED_SOURCE_SHA256

df = pd.read_csv(SOURCE_PATH)
assert df.shape == (188_318, 132)
assert df.columns.tolist() == ['id', *CATEGORICAL, *CONTINUOUS, 'loss']
assert df['id'].notna().all() and df['id'].is_unique
assert int(df.isna().sum().sum()) == 0
assert int(df.duplicated().sum()) == 0

print(f'Source verified: {len(df):,} rows x {df.shape[1]} columns')
print('Missing cells: 0; duplicate rows: 0; unique identifiers:', df['id'].nunique())

Source verified: 188,318 rows x 132 columns
Missing cells: 0; duplicate rows: 0; unique identifiers: 188318


## 2. Required recalculations

These calculations start from the raw CSV rather than copying the previously published EDA tables. Pearson and Spearman coefficients are marginal associations, not causal effects or feature-importance scores.

In [3]:
loss = df['loss']
target_summary = {
    'mean_loss': float(loss.mean()),
    'median_loss': float(loss.median()),
    'p95_loss': float(loss.quantile(0.95)),
    'max_loss': float(loss.max()),
    'raw_skewness': float(loss.skew()),
}
assert round(target_summary['mean_loss'], 2) == 3037.34
assert round(target_summary['median_loss'], 2) == 2115.57
assert round(target_summary['p95_loss'], 2) == 8508.54
assert round(target_summary['max_loss'], 2) == 121012.25
assert round(target_summary['raw_skewness'], 3) == 3.795

cardinalities = df[CATEGORICAL].nunique(dropna=False)
assert len(cardinalities) == 116
assert int((cardinalities == 2).sum()) == 72
assert int((cardinalities <= 4).sum()) == 88
assert cardinalities[['cat1', 'cat109', 'cat110', 'cat116']].to_dict() == {
    'cat1': 2, 'cat109': 84, 'cat110': 131, 'cat116': 326
}

target_correlations = pd.DataFrame({
    'predictor': CONTINUOUS,
    'pearson_loss': [df[field].corr(loss, method='pearson') for field in CONTINUOUS],
    'spearman_loss': [df[field].rank().corr(loss.rank(), method='pearson') for field in CONTINUOUS],
})
target_correlations['absolute_pearson_loss'] = target_correlations['pearson_loss'].abs()
target_correlations = target_correlations.sort_values(
    'absolute_pearson_loss', ascending=False, kind='stable'
).reset_index(drop=True)
assert target_correlations.head(3)['predictor'].tolist() == ['cont2', 'cont7', 'cont3']
assert target_correlations['absolute_pearson_loss'].max() < 0.15

matrix = df[CONTINUOUS].corr(method='pearson')
pair_rows = []
for left, right in itertools.combinations(CONTINUOUS, 2):
    value = float(matrix.loc[left, right])
    pair_rows.append({
        'field_1': left, 'field_2': right,
        'pearson_r': value, 'absolute_r': abs(value),
    })
predictor_pairs = pd.DataFrame(pair_rows).sort_values(
    'absolute_r', ascending=False, kind='stable'
).reset_index(drop=True)
assert len(predictor_pairs) == 91
assert predictor_pairs.head(3)[['field_1', 'field_2']].apply(tuple, axis=1).tolist() == [
    ('cont11', 'cont12'), ('cont1', 'cont9'), ('cont6', 'cont10')
]

bin_rows = []
for predictor in CONTINUOUS:
    bins = pd.qcut(df[predictor], q=10, labels=False, duplicates='drop')
    summary = (
        pd.DataFrame({'quantile_bin': bins, 'loss': loss})
        .groupby('quantile_bin', observed=True)['loss']
        .agg(row_count='size', mean_loss='mean', median_loss='median')
        .reset_index()
    )
    summary.insert(0, 'predictor', predictor)
    bin_rows.append(summary)
quantile_bins = pd.concat(bin_rows, ignore_index=True)
assert int(quantile_bins['row_count'].sum()) == len(df) * len(CONTINUOUS)

cont1_bins = quantile_bins.loc[quantile_bins['predictor'] == 'cont1'].reset_index(drop=True)
cont2_bins = quantile_bins.loc[quantile_bins['predictor'] == 'cont2'].reset_index(drop=True)
assert round(float(cont1_bins.loc[7, 'median_loss']), 2) == 1937.03
assert round(float(cont1_bins.loc[9, 'median_loss']), 2) == 2237.60
assert round(float(cont2_bins.iloc[0]['mean_loss']), 2) == 2120.27
assert round(float(cont2_bins.iloc[-1]['mean_loss']), 2) == 3481.26

display(pd.DataFrame([target_summary]))
display(cardinalities[['cat1', 'cat109', 'cat110', 'cat116']].rename('observed_levels'))
display(target_correlations.head())
display(predictor_pairs.head())

,mean_loss,median_loss,p95_loss,max_loss,raw_skewness
0,3037.337686,2115.57,8508.5365,121012.25,3.794958


cat1        2
cat109     84
cat110    131
cat116    326
Name: observed_levels, dtype: int64

,predictor,pearson_loss,spearman_loss,absolute_pearson_loss
0,cont2,0.141528,0.080066,0.141528
1,cont7,0.119799,0.054928,0.119799
2,cont3,0.111053,0.068353,0.111053
3,cont11,0.099806,0.054323,0.099806
4,cont12,0.098724,0.053624,0.098724


,field_1,field_2,pearson_r,absolute_r
0,cont11,cont12,0.994384,0.994384
1,cont1,cont9,0.929912,0.929912
2,cont6,cont10,0.883351,0.883351
3,cont6,cont13,0.815091,0.815091
4,cont1,cont10,0.808551,0.808551


## 3. Canonical findings register

Each row separates the measured result from its interpretation and from a possible modeling test. Review metadata is preserved only when the exact referenced evidence hash is unchanged.

In [4]:
evidence_paths = {
    'target': GATE3 / 'target_distribution_evidence/target_distribution_summary.csv',
    'source_profile': GATE3 / 'data_quality_evidence/source_profile.csv',
    'categorical': GATE3 / 'categorical-evidence/categorical_evidence_summary.csv',
    'target_correlations': GATE3 / 'data_quality_evidence/continuous_target_relationships.csv',
    'quantile_bins': GATE3 / 'data_quality_evidence/continuous_quantile_bins.csv',
    'predictor_pairs': GATE2 / 'continuous_analysis_evidence/predictor_pairs.csv',
    'overview': ROOT / 'Challenge-Project-Overview.md',
}
assert all(path.is_file() for path in evidence_paths.values())

base_review = {
    'reviewer': 'Pending independent reviewer',
    'review_decision': 'pending',
    'review_notes': 'Independent recalculation and sign-off remain pending.',
}

finding_rows = [
    {
        'finding_id': 'F-001',
        'title': 'Raw loss has a long right tail',
        'evidence_artifact': 'target_distribution_summary',
        'evidence_path': relative(evidence_paths['target']),
        'evidence_locator': 'Single summary row: mean_loss, median_loss, p95_loss, max_loss, raw_skewness',
        'observed_result': 'Mean loss is $3,037.34, median loss is $2,115.57, the 95th percentile is $8,508.54, the maximum is $121,012.25, and raw skewness is 3.795.',
        'interpretation': 'The loss distribution is strongly right-skewed, with a relatively small number of severe claims extending the upper tail.',
        'evidence_status': 'directly observed',
        'limitation_or_alternative_explanation': 'The distribution alone does not show that high-loss observations are invalid; raw-scale averages and correlations are tail-sensitive.',
        'possible_october_implication': 'Test held-out MAE and error patterns across the loss distribution, including high-cost claims, while retaining reporting in original loss units.',
    },
    {
        'finding_id': 'F-002',
        'title': 'The source table is complete and row-unique with a light cleaning burden',
        'evidence_artifact': 'source_profile',
        'evidence_path': relative(evidence_paths['source_profile']),
        'evidence_locator': 'All 132 profile rows plus id integrity and duplicate-row checks in Issue #21 notebook',
        'observed_result': 'The source has 188,318 rows and 132 columns, zero missing cells, 188,318 unique identifiers, and zero fully duplicated rows.',
        'interpretation': 'No imputation, identifier repair, or row deduplication is required before modeling.',
        'evidence_status': 'directly observed',
        'limitation_or_alternative_explanation': 'Structural cleanliness does not establish that every value is semantically valid or that preprocessing is unnecessary.',
        'possible_october_implication': 'Keep validation checks in the modeling pipeline and fit any learned preprocessing on training partitions only.',
    },
    {
        'finding_id': 'F-003',
        'title': 'Categorical predictors mix many low-cardinality and a few high-cardinality fields',
        'evidence_artifact': 'categorical_evidence_summary',
        'evidence_path': relative(evidence_paths['categorical']),
        'evidence_locator': 'Rows categorical_fields through cat109_levels',
        'observed_result': 'Among 116 categorical fields, 72 have exactly two levels and 88 have at most four; cat109, cat110, and cat116 have 84, 131, and 326 levels respectively.',
        'interpretation': 'The categorical feature set combines mostly small vocabularies with a limited high-cardinality tail.',
        'evidence_status': 'directly observed',
        'limitation_or_alternative_explanation': 'Cardinality alone does not determine usefulness, and low-support levels can make target summaries unstable.',
        'possible_october_implication': 'Compare training-fold-fitted encoders that handle rare and unseen levels; do not discard high-cardinality fields without held-out MAE evidence.',
    },
    {
        'finding_id': 'F-004',
        'title': 'Continuous predictors have weak marginal linear relationships with loss',
        'evidence_artifact': 'continuous_target_relationships',
        'evidence_path': relative(evidence_paths['target_correlations']),
        'evidence_locator': 'All 14 predictor rows; strongest absolute raw Pearson rows are cont2, cont7, and cont3',
        'observed_result': 'The largest absolute raw Pearson correlations with loss are cont2 at 0.142, cont7 at 0.120, and cont3 at 0.111; every continuous predictor is below 0.15 in absolute value.',
        'interpretation': 'No continuous predictor has a strong marginal linear association with raw loss.',
        'evidence_status': 'directly observed',
        'limitation_or_alternative_explanation': 'Marginal correlation can miss nonlinear effects, interactions, and conditional predictive value and is not an independent feature-importance measure.',
        'possible_october_implication': 'Compare linear and nonlinear models on untouched validation data rather than screening predictors solely by marginal correlation.',
    },
    {
        'finding_id': 'F-005',
        'title': 'Binned target views reveal patterns hidden by one correlation coefficient',
        'evidence_artifact': 'continuous_quantile_bins',
        'evidence_path': relative(evidence_paths['quantile_bins']),
        'evidence_locator': 'cont1 and cont2 quantile-bin rows, including row_count, mean_loss, and median_loss',
        'observed_result': 'cont1 has near-zero raw Pearson correlation (-0.010), yet its binned median loss falls to $1,937.03 in bin 7 and rebounds to $2,237.60 in bin 9; cont2 mean loss rises from $2,120.27 in its first realized bin to $3,481.26 in its last while its median does not rise monotonically throughout.',
        'interpretation': 'The binned summaries are consistent with nonlinear or non-monotonic marginal patterns that a single correlation coefficient can hide.',
        'evidence_status': 'consistent with a pattern',
        'limitation_or_alternative_explanation': 'Bins are descriptive and dataset-specific; tied values reduce cont2 to nine realized bins, and apparent shapes may change out of sample.',
        'possible_october_implication': 'Test nonlinear learners or explicitly modeled shapes using validation MAE; fit any binning or transformations on training data only.',
    },
    {
        'finding_id': 'F-006',
        'title': 'Selected continuous predictors are strongly redundant',
        'evidence_artifact': 'predictor_pairs',
        'evidence_path': relative(evidence_paths['predictor_pairs']),
        'evidence_locator': 'First three rows after sorting unique pairs by absolute Pearson correlation',
        'observed_result': 'The three strongest continuous-feature pairs are cont11-cont12 at 0.994, cont1-cont9 at 0.930, and cont6-cont10 at 0.883.',
        'interpretation': 'These pairs contain strongly overlapping linear information in the observed source.',
        'evidence_status': 'directly observed',
        'limitation_or_alternative_explanation': 'Pairwise correlation does not prove that either field is interchangeable or unnecessary in nonlinear or conditional settings.',
        'possible_october_implication': 'Compare retained-versus-reduced feature sets with held-out MAE and treat individual linear-model coefficients cautiously when correlated predictors coexist.',
    },
    {
        'finding_id': 'F-007',
        'title': 'Anonymous predictors and one supplied dataset limit interpretation and external validation',
        'evidence_artifact': 'challenge_project_overview',
        'evidence_path': relative(evidence_paths['overview']),
        'evidence_locator': 'Dataset > Key Details and repository data inventory',
        'observed_result': 'Predictors are supplied under anonymous cat* and cont* names, and the repository contains one claims CSV rather than a separate test or future-period dataset.',
        'interpretation': 'Observed statistical relationships cannot be assigned specific business or causal meanings, and this EDA cannot measure temporal or external generalization.',
        'evidence_status': 'directly observed',
        'limitation_or_alternative_explanation': 'The absence of an external dataset does not prevent creating untouched internal validation partitions in October.',
        'possible_october_implication': 'Create reproducible train/validation splits before preprocessing, report validation MAE as internal evidence, and state that future-period performance remains untested.',
    },
]

columns = [
    'finding_id', 'title', 'evidence_artifact', 'evidence_path',
    'evidence_locator', 'evidence_sha256', 'observed_result',
    'interpretation', 'evidence_status',
    'limitation_or_alternative_explanation', 'possible_october_implication',
    'owner', 'reviewer', 'review_decision', 'review_notes', 'register_version',
]

old_findings = (
    pd.read_csv(REGISTER_PATH, dtype=str, keep_default_na=False).set_index('finding_id')
    if REGISTER_PATH.is_file() else pd.DataFrame()
)

for row in finding_rows:
    row['evidence_sha256'] = sha256(ROOT / row['evidence_path'])
    row['owner'] = 'Ragib Nehal'
    row.update(base_review)
    row['register_version'] = REGISTER_VERSION
    if row['finding_id'] in old_findings.index:
        old = old_findings.loc[row['finding_id']]
        if old['evidence_sha256'] == row['evidence_sha256']:
            row['reviewer'] = old['reviewer'] or row['reviewer']
            row['review_decision'] = old['review_decision'] or 'pending'
            row['review_notes'] = old['review_notes'] or row['review_notes']
        elif old['review_decision'] == 'approved':
            row['review_notes'] = 'Evidence changed after approval; independent recalculation and re-review are required.'

findings = pd.DataFrame(finding_rows, columns=columns)
findings.to_csv(REGISTER_PATH, index=False)
display(findings[['finding_id', 'title', 'evidence_status', 'review_decision']])

,finding_id,title,evidence_status,review_decision
0,F-001,Raw loss has a long right tail,directly observed,pending
1,F-002,The source table is complete and row-unique wi...,directly observed,pending
2,F-003,Categorical predictors mix many low-cardinalit...,directly observed,pending
3,F-004,Continuous predictors have weak marginal linea...,directly observed,pending
4,F-005,Binned target views reveal patterns hidden by ...,consistent with a pattern,pending
5,F-006,Selected continuous predictors are strongly re...,directly observed,pending
6,F-007,Anonymous predictors and one supplied dataset ...,directly observed,pending


## 4. Review evidence and audit trail

The comparison status records whether this run reproduces the expected evidence. The separate review decision remains pending until a non-author performs and documents the independent run.

In [5]:
def finding_check(finding_id, headline, evidence_path, validation,
                  comparison_count, max_absolute_difference, passed):
    return {
        'finding_id': finding_id, 'headline': headline,
        'evidence_path': relative(evidence_path), 'validation': validation,
        'comparison_count': comparison_count,
        'max_absolute_difference': max_absolute_difference,
        'status': 'PASS' if passed else 'FAIL',
    }


target_evidence = pd.read_csv(evidence_paths['target']).iloc[0]
target_fields = ['mean_loss', 'median_loss', 'p95_loss', 'max_loss', 'raw_skewness']
target_differences = [
    abs(target_summary[field] - float(target_evidence[field]))
    for field in target_fields
]

source_evidence = pd.read_csv(evidence_paths['source_profile'], keep_default_na=False)
source_comparisons = [
    source_evidence['column'].tolist() == df.columns.tolist(),
    int(source_evidence['missing'].sum()) == int(df.isna().sum().sum()),
    int(source_evidence.loc[source_evidence['column'] == 'id', 'n_unique'].iloc[0])
        == int(df['id'].nunique()),
    len(source_evidence) == df.shape[1],
]

categorical_evidence = pd.read_csv(evidence_paths['categorical']).set_index('check')
categorical_expected = {
    'categorical_fields': len(cardinalities),
    'fields_with_2_levels': int((cardinalities == 2).sum()),
    'fields_with_at_most_4_levels': int((cardinalities <= 4).sum()),
    'cat116_levels': int(cardinalities['cat116']),
    'cat110_levels': int(cardinalities['cat110']),
    'cat109_levels': int(cardinalities['cat109']),
}
categorical_comparisons = [
    int(categorical_evidence.loc[name, 'observed_value']) == value
    for name, value in categorical_expected.items()
]

published_target_correlations = (
    pd.read_csv(evidence_paths['target_correlations']).set_index('predictor').loc[CONTINUOUS]
)
calculated_target_correlations = target_correlations.set_index('predictor').loc[CONTINUOUS]
correlation_differences = np.concatenate([
    np.abs(
        calculated_target_correlations[column].to_numpy(dtype=float)
        - published_target_correlations[column].to_numpy(dtype=float)
    )
    for column in ['pearson_loss', 'spearman_loss']
])

published_bins = pd.read_csv(evidence_paths['quantile_bins']).sort_values(
    ['predictor', 'quantile_bin']
).reset_index(drop=True)
calculated_bins = quantile_bins.sort_values(
    ['predictor', 'quantile_bin']
).reset_index(drop=True)
bin_keys_match = published_bins[['predictor', 'quantile_bin']].equals(
    calculated_bins[['predictor', 'quantile_bin']]
)
bin_columns = ['row_count', 'mean_loss', 'median_loss']
bin_differences = np.abs(
    calculated_bins[bin_columns].to_numpy(dtype=float)
    - published_bins[bin_columns].to_numpy(dtype=float)
)

published_pairs = pd.read_csv(evidence_paths['predictor_pairs']).sort_values(
    ['field_1', 'field_2']
).reset_index(drop=True)
calculated_pairs = predictor_pairs.sort_values(
    ['field_1', 'field_2']
).reset_index(drop=True)
pair_keys_match = published_pairs[['field_1', 'field_2']].equals(
    calculated_pairs[['field_1', 'field_2']]
)
pair_columns = ['pearson_r', 'absolute_r']
pair_differences = np.abs(
    calculated_pairs[pair_columns].to_numpy(dtype=float)
    - published_pairs[pair_columns].to_numpy(dtype=float)
)

overview_text = evidence_paths['overview'].read_text(encoding='utf-8').lower()
overview_comparisons = [
    'anonymized' in overview_text,
    len(list((ROOT / 'data').glob('*.csv'))) == 1,
]

checks = [
    finding_check('F-001', 'target tail', evidence_paths['target'],
                  'five target-summary fields compared with published evidence',
                  len(target_differences), max(target_differences),
                  np.allclose(target_differences, 0, rtol=0, atol=1e-12)),
    finding_check('F-002', 'source cleanliness', evidence_paths['source_profile'],
                  'schema, missingness, identifier uniqueness, and field count compared',
                  len(source_comparisons), 0.0, all(source_comparisons)),
    finding_check('F-003', 'categorical cardinality mixture', evidence_paths['categorical'],
                  'six inventory checks compared with published evidence',
                  len(categorical_comparisons), 0.0, all(categorical_comparisons)),
    finding_check('F-004', 'weak marginal correlations', evidence_paths['target_correlations'],
                  'Pearson and Spearman values for all 14 predictors compared',
                  correlation_differences.size, float(correlation_differences.max()),
                  np.allclose(correlation_differences, 0, rtol=0, atol=1e-12)),
    finding_check('F-005', 'binned target patterns', evidence_paths['quantile_bins'],
                  'row count, mean loss, and median loss for all 137 bins compared',
                  bin_differences.size, float(bin_differences.max()),
                  bin_keys_match and np.allclose(bin_differences, 0, rtol=0, atol=1e-10)),
    finding_check('F-006', 'continuous redundancy', evidence_paths['predictor_pairs'],
                  'both correlation fields for all 91 unique pairs compared',
                  pair_differences.size, float(pair_differences.max()),
                  pair_keys_match and np.allclose(pair_differences, 0, rtol=0, atol=1e-12)),
    finding_check('F-007', 'interpretation and validation limits', evidence_paths['overview'],
                  'anonymous-field statement and single supplied CSV confirmed',
                  len(overview_comparisons), 0.0, all(overview_comparisons)),
]
finding_checks = pd.DataFrame(checks)
checks_path = EVIDENCE_DIR / 'finding_evidence_checks.csv'
finding_checks.to_csv(checks_path, index=False)

review_rows = []

def add_review(area, metric, expected, actual, tolerance=0):
    if isinstance(expected, (int, float)) and isinstance(actual, (int, float)):
        passed = abs(float(actual) - float(expected)) <= float(tolerance)
    else:
        passed = str(actual) == str(expected)
    review_rows.append({
        'review_area': area, 'metric': metric, 'expected_value': expected,
        'recalculated_value': actual, 'tolerance': tolerance,
        'comparison_status': 'PASS' if passed else 'FAIL',
        'reviewer': 'pending', 'review_date': 'pending',
        'review_decision': 'pending',
    })

add_review('source cleanliness', 'source_rows', 188318, len(df))
add_review('source cleanliness', 'source_columns', 132, df.shape[1])
add_review('source cleanliness', 'missing_cells', 0, int(df.isna().sum().sum()))
add_review('source cleanliness', 'duplicate_rows', 0, int(df.duplicated().sum()))
add_review('source cleanliness', 'unique_ids', 188318, int(df['id'].nunique()))
add_review('target summary', 'mean_loss', 3037.34, target_summary['mean_loss'], 0.005)
add_review('target summary', 'median_loss', 2115.57, target_summary['median_loss'], 0.005)
add_review('target summary', 'p95_loss', 8508.54, target_summary['p95_loss'], 0.005)
add_review('target summary', 'max_loss', 121012.25, target_summary['max_loss'], 0.005)
add_review('target summary', 'raw_skewness', 3.795, target_summary['raw_skewness'], 0.0005)
add_review('categorical cardinality', 'categorical_fields', 116, len(cardinalities))
add_review('categorical cardinality', 'two_level_fields', 72, int((cardinalities == 2).sum()))
add_review('categorical cardinality', 'four_or_fewer_level_fields', 88, int((cardinalities <= 4).sum()))
for field, expected in {'cat1': 2, 'cat109': 84, 'cat110': 131, 'cat116': 326}.items():
    add_review('categorical cardinality', f'{field}_levels', expected, int(cardinalities[field]))
add_review('continuous-to-target correlations', 'continuous_target_rows', 14, len(target_correlations))
for predictor in CONTINUOUS:
    for column in ['pearson_loss', 'spearman_loss']:
        add_review(
            'continuous-to-target correlations', f'{predictor}_{column}',
            float(published_target_correlations.loc[predictor, column]),
            float(calculated_target_correlations.loc[predictor, column]), 1e-12,
        )
for rank, row in target_correlations.head(3).iterrows():
    add_review('continuous-to-target correlations', f'target_rank_{rank + 1}',
               ['cont2', 'cont7', 'cont3'][rank], row['predictor'])
add_review('continuous-feature pairs', 'continuous_pair_rows', 91, len(predictor_pairs))
for rank, row in predictor_pairs.head(3).iterrows():
    add_review('continuous-feature pairs', f'pair_rank_{rank + 1}',
               ['cont11-cont12', 'cont1-cont9', 'cont6-cont10'][rank],
               f"{row['field_1']}-{row['field_2']}")
headline_metrics = {
    'source_cleanliness_supported': 'F-002',
    'target_tail_supported': 'F-001',
    'categorical_mixture_supported': 'F-003',
    'weak_correlations_supported': 'F-004',
    'continuous_redundancy_supported': 'F-006',
}
finding_status = finding_checks.set_index('finding_id')['status']
for metric, finding_id in headline_metrics.items():
    add_review('headline finding', metric, 'PASS', finding_status.loc[finding_id])

review_results = pd.DataFrame(review_rows)
assert review_results['comparison_status'].eq('PASS').all()
recalculation_path = EVIDENCE_DIR / 'independent_recalculation_results.csv'
review_results.to_csv(recalculation_path, index=False)

review_columns = [
    'issue', 'artifact', 'artifact_path', 'artifact_sha256', 'author',
    'independent_reviewer', 'review_date', 'review_decision',
    'fresh_run_verified', 'disagreements', 'limitations', 'review_notes',
]
reviews = pd.read_csv(REVIEW_REGISTER_PATH, dtype=str, keep_default_na=False)
issue21 = {
    'issue': '#21', 'artifact': 'findings_register',
    'artifact_path': relative(REGISTER_PATH),
    'artifact_sha256': sha256(REGISTER_PATH), 'author': 'Ragib Nehal',
    'independent_reviewer': '', 'review_date': '',
    'review_decision': 'pending', 'fresh_run_verified': 'pending',
    'disagreements': '',
    'limitations': 'Independent recalculation by a named non-author is pending.',
    'review_notes': 'Automated reference comparisons pass; human sign-off is not yet complete.',
}
existing_issue21 = reviews.loc[reviews['issue'] == '#21']
if not existing_issue21.empty:
    old = existing_issue21.iloc[0]
    if old['artifact_sha256'] == issue21['artifact_sha256']:
        for field in ['independent_reviewer', 'review_date', 'review_decision',
                      'fresh_run_verified', 'disagreements', 'limitations', 'review_notes']:
            issue21[field] = old[field] or issue21[field]
    reviews = reviews.loc[reviews['issue'] != '#21']
reviews = pd.concat([reviews, pd.DataFrame([issue21])], ignore_index=True)
reviews[review_columns].to_csv(REVIEW_REGISTER_PATH, index=False)

base_git_revision = subprocess.run(
    ['git', 'rev-parse', 'HEAD'], cwd=ROOT, check=True, capture_output=True, text=True
).stdout.strip()
working_tree_dirty = bool(subprocess.run(
    ['git', 'status', '--porcelain'], cwd=ROOT, check=True, capture_output=True, text=True
).stdout.strip())
artifacts = []
for path in [REGISTER_PATH, checks_path, recalculation_path]:
    if path.suffix == '.csv':
        frame = pd.read_csv(path, keep_default_na=False)
        schema = frame.columns.tolist()
        row_count = len(frame)
    artifacts.append({
        'path': relative(path), 'sha256': sha256(path),
        'row_count': row_count, 'columns': schema,
    })
manifest = {
    'issue': '#21', 'status': 'ready for independent review',
    'workflow_run_id': f'issue21-{source_sha256[:12]}-v2',
    'base_git_revision': base_git_revision,
    'working_tree_dirty': working_tree_dirty,
    'notebook': {
        'path': relative(NOTEBOOK_PATH),
        'executable_source_sha256': executable_notebook_sha256(NOTEBOOK_PATH),
    },
    'source': {
        'path': relative(SOURCE_PATH), 'sha256': source_sha256,
        'rows': len(df), 'columns': df.shape[1],
    },
    'runtime': {
        'python': platform.python_version(),
        'pandas': pd.__version__, 'numpy': np.__version__,
    },
    'configuration': {
        'random_seed': None, 'requested_quantile_bins': 10,
        'quantile_duplicate_rule': 'drop',
        'categorical_support_threshold_rows': 188,
    },
    'review': {
        'reviewer': 'pending', 'review_date': 'pending',
        'review_decision': 'pending',
    },
    'artifacts': artifacts,
}
manifest_path = EVIDENCE_DIR / 'run_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2) + '\n', encoding='utf-8')

display(finding_checks)
display(review_results)

,finding_id,headline,evidence_path,validation,comparison_count,max_absolute_difference,status
0,F-001,target tail,notebooks/final-deliverables/September/Gate 3/...,five target-summary fields compared with publi...,5,0.000000e+00,PASS
1,F-002,source cleanliness,notebooks/final-deliverables/September/Gate 3/...,"schema, missingness, identifier uniqueness, an...",4,0.000000e+00,PASS
2,F-003,categorical cardinality mixture,notebooks/final-deliverables/September/Gate 3/...,six inventory checks compared with published e...,6,0.000000e+00,PASS
3,F-004,weak marginal correlations,notebooks/final-deliverables/September/Gate 3/...,Pearson and Spearman values for all 14 predict...,28,9.714451e-17,PASS
4,F-005,binned target patterns,notebooks/final-deliverables/September/Gate 3/...,"row count, mean loss, and median loss for all ...",411,4.547474e-13,PASS
5,F-006,continuous redundancy,notebooks/final-deliverables/September/Gate 2/...,both correlation fields for all 91 unique pair...,182,2.220446e-16,PASS
6,F-007,interpretation and validation limits,Challenge-Project-Overview.md,anonymous-field statement and single supplied ...,2,0.000000e+00,PASS


,review_area,metric,expected_value,recalculated_value,tolerance,comparison_status,reviewer,review_date,review_decision
0,source cleanliness,source_rows,188318,188318,0.000000e+00,PASS,pending,pending,pending
1,source cleanliness,source_columns,132,132,0.000000e+00,PASS,pending,pending,pending
2,source cleanliness,missing_cells,0,0,0.000000e+00,PASS,pending,pending,pending
3,source cleanliness,duplicate_rows,0,0,0.000000e+00,PASS,pending,pending,pending
4,source cleanliness,unique_ids,188318,188318,0.000000e+00,PASS,pending,pending,pending
5,target summary,mean_loss,3037.34,3037.337686,5.000000e-03,PASS,pending,pending,pending
6,target summary,median_loss,2115.57,2115.57,5.000000e-03,PASS,pending,pending,pending
7,target summary,p95_loss,8508.54,8508.5365,5.000000e-03,PASS,pending,pending,pending
8,target summary,max_loss,121012.25,121012.25,5.000000e-03,PASS,pending,pending,pending
9,target summary,raw_skewness,3.795,3.794958,5.000000e-04,PASS,pending,pending,pending


## 5. Final automated verification

The checks below validate structure, evidence hashes, wording boundaries, and deterministic artifact schemas.

In [6]:
published_findings = pd.read_csv(REGISTER_PATH, dtype=str, keep_default_na=False)
assert published_findings['finding_id'].tolist() == [f'F-{i:03d}' for i in range(1, 8)]
assert published_findings['finding_id'].is_unique
assert published_findings['register_version'].eq(REGISTER_VERSION).all()
assert published_findings['review_decision'].isin({
    'pending', 'approved', 'changes requested'
}).all()
approved_findings = published_findings['review_decision'].eq('approved')
assert published_findings.loc[approved_findings, 'reviewer'].ne('').all()
assert published_findings['evidence_status'].isin({
    'directly observed', 'consistent with a pattern', 'hypothesis', 'inconclusive'
}).all()
for _, row in published_findings.iterrows():
    evidence_path = ROOT / row['evidence_path']
    assert evidence_path.is_file()
    assert row['evidence_sha256'] == sha256(evidence_path)

modeling_terms = ('test ', 'compare ', 'fit ', 'create reproducible')
assert published_findings['possible_october_implication'].str.lower().apply(
    lambda value: any(term in value for term in modeling_terms)
).all()
assert not published_findings['observed_result'].str.lower().str.contains(
    r'causes?|feature importance|held-out performance', regex=True
).any()
assert finding_checks['status'].eq('PASS').all()
assert review_results['comparison_status'].eq('PASS').all()
assert review_results['review_decision'].eq('pending').all()
assert sorted(path.name for path in EVIDENCE_DIR.iterdir() if path.is_file()) == [
    'finding_evidence_checks.csv',
    'independent_recalculation_results.csv',
    'run_manifest.json',
]

print('Issue #21 automated verification: PASS')
print('Status: READY FOR INDEPENDENT REVIEW')
print('Independent reviewer, review date, fresh run, and approval: PENDING')

Issue #21 automated verification: PASS
Status: READY FOR INDEPENDENT REVIEW
Independent reviewer, review date, fresh run, and approval: PENDING


# Independent recalculation pending

The automated source-to-evidence comparisons pass, but Issue #21 is **not complete**. A named reviewer who did not author this EDA must rerun the notebook from a fresh kernel, independently inspect the target summary, representative categorical cardinalities including `cat116`, all continuous-to-target correlations, the three strongest continuous-feature pairs, and at least five headline finding checks.

## Reviewer rerun and sign-off steps

1. Confirm you are reviewing a clean, committed project version and that `data/allstate_claims_data.csv` is present. Record the commit being reviewed.
2. Open this notebook, restart the kernel, and run every cell from top to bottom. Do not reuse values from the author's saved outputs.
3. Confirm the final code cell prints `Issue #21 automated verification: PASS` and that every row in `finding_evidence_checks.csv` and `independent_recalculation_results.csv` has comparison status `PASS`.
4. Independently inspect the recalculated target summary; cardinalities for `cat1`, `cat109`, `cat110`, and `cat116`; Pearson and Spearman correlations for all 14 continuous fields; the three strongest continuous-feature pairs; and at least five headline findings. Record any disagreement instead of changing a calculated value to force agreement.
5. From the repository root, activate whichever Python environment has the project dependencies installed, then run the artifact regression test separately from the notebook:

   ```bash
   python -m unittest tests/test_issue_21_artifacts.py -v
   ```

   Use the Python interpreter for the environment in which the project dependencies are installed. If your system exposes Python 3 as `python3`, substitute `python3` for `python`. A virtual environment is recommended for reproducibility but is not required. Optional environment-specific commands are `.venv/bin/python -m unittest tests/test_issue_21_artifacts.py -v` on macOS/Linux and `.venv\Scripts\python -m unittest tests/test_issue_21_artifacts.py -v` on Windows. A passing notebook does not run this test automatically.
6. After all checks finish, update the Issue `#21` row in `independent_review_register.csv` with your name, review date, `fresh_run_verified`, decision, disagreements, limitations, and review notes.
7. Replace the pending reviewer metadata in `independent_recalculation_results.csv` and the `review` object in `run_manifest.json`. Update the Gate 3 report's reviewer fields, verification checkboxes, review notes, and status. If changes are requested, leave the issue open and describe them precisely.
8. Perform these metadata edits only after the final notebook rerun because rerunning this authoring notebook republishes the evidence bundle with pending review metadata. Re-run the artifact test after the edits; do not rerun the notebook again unless the analysis itself changes.

Until a named non-author completes and records these steps, the correct status is **Ready for independent review**, not Complete.